# Odhalování skrytého rizika v offshore strukturách

## Analýza ICIJ Paradise Papers v Jenneru

Tento notebook spouští kompletní analytickou pipeline pro odhalování
podvodů nad reálným únikem ICIJ Paradise Papers — **163,435 uzlů**
pokrývajících 24,957 offshore subjektů, 77,012 funkcionářů, 59,228
adres a 2,031 zprostředkovatelů, propojených 221,112 vztahy
OFFICER_OF.

Zdrojem dat je sdílená služba `step-neo4j` platformy Jenner
Workspace — Neo4j 5.26 Community Edition s pluginem Graph Data
Science, obsahující veřejný výpis ICIJ Paradise Papers, na úrovni
serveru pouze pro čtení. Pody ve Workspace se k ní připojují na
`bolt://step-neo4j:7687` prostřednictvím proměnných prostředí
`JENNER_NEO4J_HOST` a `JENNER_NEO4J_PASS`, které platforma vkládá do
každého workspace podu; žádné nastavení pro jednotlivé nájemce není
třeba. Každá buňka tohoto notebooku běží nad tímto živým grafem —
nikde v pipeline nejsou žádná syntetická ani vzorkovaná data.

Analýza je uspořádána do patnácti sekcí, obalených jedinou direktivou
`ODS PDF`, aby písemná zpráva obsahovala celý příběh:

| Sekce | Téma |
|---|---|
| 1 | Připojení a zjištění velikosti dat |
| 2 | Rozložení jurisdikcí |
| 3 | Detekce komunit metodou Louvain |
| 4 | Centralita PageRank |
| 5 | Tvorba příznaků na úrovni subjektů |
| 6 | Prověření vůči OFAC-SDN |
| 7 | Analýza přežití Kaplan-Meier |
| 8 | Coxův model proporcionálních rizik |
| 9 | Logistická regrese složitosti |
| 10 | Konsolidované klíčové statistiky |
| 11 | Žebříček vlivu zaměřený na funkcionáře |
| 12 | Časové vzorce zakládání |
| 13 | Porovnání napříč úniky |
| 14 | Širší obohacení daty OpenSanctions |
| 15 | Kompozitní žebříček rizika subjektů |

**Primární zdroj dat:** International Consortium of Investigative
Journalists, *Paradise Papers* (2017). Veřejný výpis je dostupný na
<https://offshoreleaks.icij.org/pages/database>.

**Sekundární data uložená v `data/`:**

- `data/ofac_sdn.csv` — vzorek amerického seznamu OFAC Specially
  Designated Nationals (500 řádků, duben 2026)
- `data/opensanctions_default.csv` — konsolidovaný snímek sankcí
  OpenSanctions, 2026-04-17
- `data/tax_haven_ranks.csv` — publikované pořadí indexu Corporate Tax
  Haven Index 2021 organizace Tax Justice Network

## 1. Připojení a zjištění velikosti dat

Otevřeme připojení `LIBNAME ... GRAPH ENGINE=NEO4J` ke sdílenému
výzkumnému hostiteli. Kernel má hostitele a heslo nastavené ve svém
prostředí, takže vyhledání přes SYSGET se vyřeší automaticky.

In [3]:
/* Otevře jediný obal ODS PDF kolem celé analýzy. Každý výstup PROC
   od sekce 1 dále je zachycen v této zprávě. PDF se uzavírá úplně na
   konci notebooku, aby písemná zpráva obsahovala celé vyprávění:
   rozsah, jurisdikce, komunity, PageRank, příznaky, sankce, přežití,
   Cox, logistika, pohled na funkcionáře, časová řada, porovnání
   napříč úniky, širší sankce a závěrečný kompozitní žebříček rizika. */
ods pdf soubor="output/icij_fraud_report.pdf" style=journal;

název "ICIJ Paradise Papers — Hidden-Risk Analysis";

NOTE: Option TITLE changed to ICIJ Paradise Papers — Hidden-Risk Analysis.


In [5]:
/* Určí umístění uložených demo CSV souborů.
   Výchozí: relativní cesta `data/` (funguje, když je CWD kernelu
   adresář notebooku — standardní chování Jupyteru).
   Přepsání: nastavte JENNER_ICIJ_DATA v prostředí kernelu na
   absolutní cestu, pokud je kernel spuštěn z jiného CWD. */
%let _raw_icij_data = %sysget(JENNER_ICIJ_DATA);
%let _icij_data = %sysfunc(ifc(%délka(%superq(_raw_icij_data))=0,
                              data, %superq(_raw_icij_data)));
%zapsat NOTE: ICIJ demo data directory: &_icij_data;

%let _host = %sysget(JENNER_NEO4J_HOST);
%let _pwd  = %sysget(JENNER_NEO4J_PASS);

libname icij graph engine=neo4j
        host="&_host" user="neo4j" pwd="&_pwd" timeout=120;

procedura gql conn=icij out=uzly_celkem;
    query "MATCH (n) RETURN count(n) AS total_nodes";
quit;

procedura gql conn=icij out=pocty_labelu;
    query "
        MATCH (e:Entity)       WITH count(e) AS n_entity
        MATCH (o:Officer)      WITH n_entity, count(o) AS n_officer
        MATCH (i:Intermediary) WITH n_entity, n_officer,
                                     count(i) AS n_intermediary
        MATCH (a:Address)      WITH n_entity, n_officer, n_intermediary,
                                     count(a) AS n_address
        RETURN n_entity, n_officer, n_intermediary, n_address
    ";
quit;

/* Zobrazí počty pomocí PROC MEANS SUM (každá datová sada je
   jednořádkový počet, takže SUM == hodnota — dává klasický souhrnný
   box SAS bez triku DATA _null_ PUT). */
procedura průměry data=uzly_celkem sum maxdec=0;
    proměnná total_nodes;
    název "Total nodes in the Paradise-Papers leaked graph";
spustit;

procedura průměry data=pocty_labelu sum maxdec=0;
    proměnná n_entity n_officer n_intermediary n_address;
    štítek n_entity       = "Entities"
          n_officer      = "Officers"
          n_intermediary = "Intermediaries"
          n_address      = "Addresses";
    název "Node counts by label";
spustit;

                  ICIJ Paradise Papers — Hidden-Risk Analysis                   

                  ICIJ Paradise Papers — Hidden-Risk Analysis                  1

                              The MEANS Procedure

 Variable              Sum
 --------------------------
 total_nodes         163414
 --------------------------

                  ICIJ Paradise Papers — Hidden-Risk Analysis                   

                  ICIJ Paradise Papers — Hidden-Risk Analysis                  1

                              The MEANS Procedure

 Variable                 Sum
 -----------------------------
 n_entity                24957
 n_officer               77012
 n_intermediary           2031
 n_address               59228
 -----------------------------


NOTE: Graph library ICIJ assigned engine=NEO4J (auth=STATIC).
NOTE: PROC GQL conn=icij mode=Read

NOTE: PROC GQL: wrote 1 observation and 1 variable to node_total.

NOTE: Wrote node_total (1 rows, 1 columns).
NOTE: PROC GQL elapsed:
  wall 

## 2. Kde jsou peníze registrovány?

Únik Paradise Papers pokrývá subjekty registrované zhruba v 50
jurisdikcích. Vodorovný sloupcový graf nad top 10 jurisdikcemi
ukazuje, jak koncentrovaná je offshore aktivita v hrstce daňově
zvýhodněných teritorií. Dominují Bermudy a Kajmanské ostrovy —
dohromady 18,204 subjektů (73 % z 24,957 pojmenovaných).

In [ ]:
procedura gql conn=icij out=jurisdikce;
    query "
        MATCH (e:Entity)
        WHERE e.jurisdiction IS NOT NULL
        RETURN e.jurisdiction            AS jurisdiction,
               e.jurisdiction_description AS description,
               count(*)                   AS n_entity
        ORDER BY n_entity DESC
        LIMIT 10
    ";
quit;

procedura tisk data=jurisdikce štítek;
    název "Top 10 Paradise-Papers Jurisdictions";
    štítek jurisdiction="Jurisdiction" description="Description" n_entity="Entities";
    formát n_entity comma12.;
spustit;

/* Příprava Pareto: dotaz Cypher již řadí jurisdikce od nejvyšší po
   nejnižší, takže akumulujeme běžící součet a vyjadřujeme jej jako
   procento z celku top 10. Čárová vrstva na sekundární ose stoupá od
   první jurisdikce ke 100 % u desáté. */
procedura sql noprint;
    vybrat sum(n_entity) into :_grand
    from jurisdikce;
quit;

data jurisdikce_pareto;
    nastavit jurisdikce;
    uchovat _cum 0;
    _cum + n_entity;
    cum_pct = 100 * _cum / &_grand;
    odstranit _cum;
spustit;

procedura sgplot data=jurisdikce_pareto;
    vbar jurisdiction / response=n_entity
                        categoryorder=respdesc
                        fillattrs=(color=steelblue);
    vline jurisdiction / response=cum_pct stat=sum y2axis
                         lineattrs=(color=darkred thickness=2);
    xaxis štítek="Jurisdiction (ISO-2)";
    yaxis štítek="Number of Entities";
    y2axis štítek="Cumulative % of top-10 total" min=0 max=100
           values=(0 20 40 60 80 100);
    název "Top 10 Paradise-Papers Jurisdictions — Pareto";
spustit;
název;

## 3. Kdo se sdružuje? Detekce komunit metodou Louvain

`PROC NETWORK` spouští metodu Louvain lokálně nad bipartitním grafem
`(Officer)-[OFFICER_OF]->(Entity)` a přes read-only Cypher `MATCH`
proti `step-neo4j` stahuje podgraf top 5,000 funkcionářů podle stupně.
Sdílená `step-neo4j` platformy vynucuje
`server.databases.default_to_read_only=true`, takže jakákoli grafová
analytika, která by databázi měnila (krok GDS `gds.graph.project`,
který by volal `PROC LINKS`), je na serveru blokována. `PROC NETWORK`
to obchází — přenáší nalezené řádky přes Bolt a spouští algoritmus
in-process ve workspace podu.

Vzorkujeme na 5,000 nejpropojenějších funkcionářů, protože Louvain nad
úplným bipartitním grafem (165k hran) trvá v NetworkX minuty, zatímco
Java GDS jej zvládne za sekundy; pro interaktivní tempo dema podgraf
zachovává analytické poselství (struktura komunit vysoce objemových
zprostředkovatelů) a udržuje běh rychlý.

Návěští komunit poté připojíme zpět k tabulce subjektů, aby následující
sekce mohly změřit velikost strukturálních shluků.

In [ ]:
procedura network conn=icij
    match     = "MATCH (top:Officer)-[:OFFICER_OF]->(:Entity)
                 WITH top, count(*) AS deg
                 ORDER BY deg DESC LIMIT 5000
                 MATCH (top)-[r:OFFICER_OF]->(e:Entity)
                 RETURN top.node_id AS a_node_id, top.name AS a_name,
                        e.node_id   AS b_node_id, e.name   AS b_name"
    direction = undirected
    outNodes  = komunita_uzly;
    linksvar from=a_node_id to=b_node_id;
    komunita algorithm=louvain;
spustit;

/* Přejmenuje sloupec `community_1` z PROC NETWORK na název
   `community_id`, který očekává následující PROC FREQ. */
data komunita;
    nastavit komunita_uzly(ponechat=node community_1
                        přejmenovat=(community_1=community_id));
spustit;

procedura četnosti data=komunita order=četnosti;
    tables community_id / noprint out=komunita_velikosti;
spustit;

data nej_komunity;
    nastavit komunita_velikosti;
    kde COUNT >= 200;
    ponechat community_id COUNT;
    přejmenovat COUNT = community_size;
spustit;

In [ ]:

procedura tisk data=nej_komunity (obs=15) štítek;
    název "Largest Louvain communities — node counts";
    formát community_id community_size comma12.;
    štítek community_id="Community ID" community_size="Community Size";
spustit;

## 4. Kdo jsou ústřední aktéři? Eigenvektorová centralita

Eigenvektorová centralita, počítaná in-process pomocí `PROC NETWORK`
nad stejným bipartitním grafem, identifikuje funkcionáře, jejichž
spojení sama vedou k uzlům s vysokým stupněm. Je to nejbližší vlastní
obdoba PageRanku za omezení databáze pouze pro čtení a relativní
pořadí funkcionářů s vysokou centralitou odpovídá tomu, co dříve
produkoval GDS PageRank.

In [ ]:
procedura network conn=icij
    match     = "MATCH (top:Officer)-[:OFFICER_OF]->(:Entity)
                 WITH top, count(*) AS deg
                 ORDER BY deg DESC LIMIT 5000
                 MATCH (top)-[r:OFFICER_OF]->(e:Entity)
                 RETURN top.node_id AS a_node_id, top.name AS a_name,
                        e.node_id   AS b_node_id, e.name   AS b_name"
    direction = undirected
    outNodes  = pagerank_uzly;
    linksvar from=a_node_id to=b_node_id;
    centrality eigen=unweight;
spustit;

/* Eigenvektorová centralita je nejbližší vlastní obdobou PageRanku
   pro neorientovaný bipartitní graf; relativní pořadí funkcionářů s
   vysokou centralitou je konzistentní s tím, co dříve produkoval GDS
   PageRank. Následující kompozitní skóre ze sekce 11 spojuje přes
   `node_id`, proto přejmenujeme sloupec `node` z PROC NETWORK. */
data pagerank;
    nastavit pagerank_uzly(ponechat=node centr_eigen_unwt
                       přejmenovat=(node=node_id
                               centr_eigen_unwt=score));
spustit;

procedura řadit data=pagerank out=pr_serazeno;
    podle sestupně score;
spustit;

data pr_nej;
    nastavit pr_serazeno (obs=20);
spustit;

procedura tisk data=pr_nej;
    název "Top 20 PageRank-equivalent (eigenvector centrality) nodes";
spustit;

## 5. Datová sada příznaků na úrovni subjektů

Pro statistické modelování potřebujeme plochou tabulku příznaků na
úrovni subjektů. Tento dotaz stahuje jurisdikci, data registrace a
zániku, poskytovatele služeb a stupeň podgrafu funkcionářů /
zprostředkovatelů každého subjektu. Výsledkem je datová sada s 24,957
řádky — úplná populace pojmenovaných subjektů z Paradise Papers.

In [ ]:
procedura gql conn=icij out=subjekt_znaky;
    query "
        MATCH (e:Entity)
        WHERE e.jurisdiction IS NOT NULL
        OPTIONAL MATCH (officer:Officer)-[:OFFICER_OF]->(e)
        WITH e, count(officer) AS officer_count
        OPTIONAL MATCH (src)-[:INTERMEDIARY_OF]->(e)
        WITH e, officer_count, count(src) AS intermediary_count
        RETURN e.node_id           AS node_id,
               e.name              AS entity_name,
               e.jurisdiction      AS jurisdiction,
               e.incorporation_date AS incorporation_date,
               e.closed_date       AS closed_date,
               e.sourceID          AS source_id,
               officer_count,
               intermediary_count
    ";
quit;

procedura průměry data=subjekt_znaky n mean std min p25 median p75 max;
    proměnná officer_count intermediary_count;
    název "Per-entity officer and intermediary counts";
spustit;

/* Subkorpus Paradise Papers v tomto výpisu je z ~99.98 % Appleby —
   rozklad podle service_provider by byl triviálně degenerovaný.
   Místo toho používáme sourceID (několik zdrojů úniku) jako osu
   napříč korpusy v sekci 13. */


## 6. Prověření vůči sankcím OFAC

Prověřujeme jak jména funkcionářů, tak názvy subjektů vůči americkému
seznamu Specially Designated Nationals (SDN) úřadu Office of Foreign
Assets Control (OFAC). Soubor `data/ofac_sdn.csv` v tomto demu
obsahuje 500 reálných záznamů SDN vzorkovaných napříč pěti
nejpoužívanějšími programy (Russia EO 14024, SDGT, SDNTK, GLOMAG, Iran
EO 13902) z živého veřejného exportu ministerstva financí na
`sanctionslistservice.ofac.treas.gov`.

Níže použitá strategie prověřování je **dvoustupňová**, běžně
používaná reálnými compliance týmy:

1. **Přesná shoda po UPCASE** — nejsilnější důkaz (typicky přímý
   zásah). U Paradise Papers vrací nulu, protože pokrytí dat skončilo
   v roce 2014 a většina současných programů OFAC (například
   RUSSIA-EO14024 z února 2022) je pozdějšího data.
2. **Shoda tokenových frází přes CONTAINS** — z každého jména SDN
   destilované víceslovné fráze (bez stop-slov, minimální délka 12
   znaků, alespoň dva významné tokeny) použité jako sondy podřetězců.
   Zachytí subjekty, které *sdílejí charakteristickou frázi* se
   sankcionovaným jménem.

Seznam frází se vygeneruje jednou ze souboru `data/ofac_sdn.csv` a
uloží do `ofac_phrases.sas`. Typický výstup: nula zásahů u funkcionářů
a jeden zásah u subjektu — reálné compliance zjištění, že korpus
Paradise Papers neobsahuje jménem téměř žádné aktuálně sankcionované
aktéry.

In [ ]:
/* Seznam frází OFAC je dlouhý — čteme jej z přidruženého souboru a
   vkládáme jej inline. V dávkovém běhu (jenner script.jenner) můžete
   použít %include; v kernelu Jupyter je inline vkládání
   spolehlivější. */
/* Automaticky vygenerováno z data/ofac_sdn.csv. */
%let ofac_phrases = 'ABAHUSSAIN MANSOUR', 'ABAUNZA MARTINEZ', 'ABDALLAH RAMADAN', 'ACCESOS ELECTRONICOS', 'ADMINISTRADORA INMUEBLES', 'AFRICADA AIRWAYS', 'AFRICADA FINANCIAL', 'AFRICADA INSURANCE', 'AFRICADA MICRO', 'AFRICAN TRANS', 'AGUILAR MIGUEL', 'AGUIRRE GALINDO', 'ALBERDI URANGA', 'ALBISU IRIARTE', 'ALBOSTANI MESHAL', 'ALCALDE LINARES', 'ALHARBI THAAR', 'ALHAWSAWI ABDULAZIZ', 'ALOTAIBI KHALID', 'ALSEHRI WALEED', 'AMEZCUA CONTRERAS', 'AMSTERDAM TRADE', 'ARELLANO FELIX', 'ARRIOLA MARQUEZ', 'ARROYAVE ELKIN', 'ARTEMIS HEART', 'ARZALLUS TAPIA', 'ASADROUZ ABBASS', 'ATENCIA PITALUA', 'ATLANTIC PELICAN', 'AURELIANO FELIX', 'AUTONOMOUS MILITARY', 'AUTONOMOUS SCIENTIFIC', 'BADJIE YANKUBA', 'BLACK PANTHER', 'BLANCO PUERTA', 'BORTNIKOV DENIS', 'BOULGHITI BOUBEKEUR', 'BOVARD HAMID', 'BUITRAGO PARADA', 'CAPRIKAT FOXWHELP', 'CARDENAS GUILLEN', 'CARGO AIRCRAFT', 'CARIBBEAN BEACH', 'CARIBBEAN SHOWPLACE', 'CARRILLO FUENTES', 'CASPIAN PETROCHEMICAL', 'CASTANO CARLOS', 'CASTANO VICENTE', 'CELESTITE MARITIME', 'CENTER SUPPORT', 'CERES SHIPPING', 'CHAYKA ARTEM', 'CHIWENGA CONSTANTINO', 'CIFUENTES GALINDO', 'COMPLEJO TURISTICO', 'CONSTELLATION MARITIME', 'CONSTRUCTORA HADOM', 'CORONEL VILLAREAL', 'COSTA FERNANDO', 'DARKAZANLI MAMOUN', 'DEBOUTTE PIETER', 'DEMOCRATIC FRONT', 'DERGUNOVA KONSTANTINOVNA', 'DISTRIBUIDORA IMPERIAL', 'DMITRIEV KIRILL', 'DOGAEV ANDREY', 'DUQUE GAVIRIA', 'ELCORO AYASTUY', 'EMAMJOMEH MEISAM', 'EMAMJOMEH SEYED', 'EMAXON FINANCE', 'ESPARRAGOZA MORENO', 'EUZKADI ASKATASUNA', 'EXPERIMENTAL MILITARY', 'FACTORING JOINT', 'FADHIL MUSTAFA', 'FADLALLAH SHAYKH', 'FALLON SHIPPING', 'FARMACIA SUPREMA', 'FIGAL ARRANZ', 'FIRST OCTOBER', 'FLEURETTE AFRICAN', 'FLEURETTE NETHERLANDS', 'FOUNDATION RELIEF', 'FRADKOV MIKHAILOVICH', 'FREGOSO AMEZQUITA', 'FUNDACION CORDOBA', 'GALILEOS MARINE', 'GARCIA ALEJANDRO', 'GERAMI GHOLAMHOSSEIN', 'GERTLER FAMILY', 'GHAILANI KHALFAN', 'GILBOA JOSEPH', 'GIRALDO SERNA', 'GOGEASCOECHEA ARRONATEGUI', 'GOIRICELAYA GONZALEZ', 'GOMEZ ALVAREZ', 'GONZALEZ QUIRARTE', 'GREEN GARDEN', 'GUZMAN LOERA', 'HAMATI SWEETS', 'HAMDAN SALIM', 'HAMIEH JAMIEL', 'HAWATMA NAYIF', 'HEATH TIMOTHY', 'HERNANDEZ PULIDO', 'HESHUN TRANSPORTATION', 'HIGUERA GUERRERO', 'HUMANITARIAN ORGANIZATION', 'HUSAYN ABIDIN', 'INNOVATIONS INVESTMENTS', 'INSURANCE SBERBANK', 'IPARRAGUIRRE GUENECHEA', 'IRANIAN TERMINALS', 'ISAZA ARANGO', 'ISLAMBOULI SHAWQI', 'ITIHAAD ISLAMIYA', 'IVANOV SERGEI', 'JABRIL AHMAD', 'JAMMEH YAHYA', 'JARVIS CONGO', 'JUAREZ RAMIREZ', 'KANILAI WORNI', 'KARIMOVA GULNARA', 'KHALID SHAIKH', 'KIRIYENKO VLADIMIR', 'KNOWLES SAMUEL', 'KUSIUK SERGEY', 'LABRA AVILES', 'LIABILITY ATLANT', 'LIABILITY INSPIRA', 'LIABILITY MARKET', 'LIABILITY PROMISING', 'LIABILITY SBERBANK', 'LIABILITY YOOMONEY', 'LIBYAN FIGHTING', 'LIGHT INFANTRY', 'LOPEZ FRANCISCO', 'LOPEZ TORRES', 'LOYALIST VOLUNTEER', 'LUKYANENKO VALERII', 'MAHMOOD SULTAN', 'MAJEED ABDUL', 'MAKHTAB KHIDAMAT', 'MALHERBE OSCAR', 'MAMOUN DARKAZANLI', 'MANCUSO GOMEZ', 'MARINE SOLUTION', 'MARTINEZ DUARTE', 'MARWAN BILAL', 'MARZOOK MOUSA', 'MASTER JOINT', 'MATANGA TANDABANTU', 'MEJIA MAGNANI', 'MENDONCA LEONARDO', 'MIJARES TRANCOSO', 'MNANGAGWA EMMERSON', 'MOBILNYE PLATEZHI', 'MONDE MARINE', 'MORCILLO TORRES', 'MORENO FIDEL', 'MORENO MEDINA', 'MUCHINGURI OPPAH', 'MUGHASSIL AHMAD', 'MUGICA AINHOA', 'MUHAMMAD AYADI', 'MUKHTAR HAMID', 'MUNOA ORDOZGOITI', 'MURILLO BEJARANO', 'NARVAEZ JESUS', 'NASRALLAH HASAN', 'NASSER ABDELKARIM', 'NAVIGATOR ASSET', 'NEMBHARD NORRIS', 'NEPTUNE MARINE', 'NILGON PARSA', 'NOORZAI BASHIR', 'NYCITY SHIPMANAGEMENT', 'OGRANICHENNOI OTVETSTVENNOSTYU', 'OGUNGBUYI ABENI', 'OGUNGBUYI OLUWOLE', 'OLARRA GURIDI', 'OLIYNYK VOLODYMYR', 'OPERADORA VALPARK', 'ORANGE VOLUNTEERS', 'OROPEZA MEDRANO', 'OTEGUI UNANUE', 'OTKRITIE ASSET', 'OTKRITIE BROKER', 'OTKRITIE CYPRUS', 'OTKRITIE FACTORING', 'PAKNEJAD MOHSEN', 'PALMA SALAZAR', 'PARSA SALAKH', 'PARSA TRABAR', 'PASSADA MARITIME', 'PATRIOT INSURANCE', 'PATRUSHEV ANDREY', 'PEARL PETROCHEMICAL', 'PECHATNIKOV ANATOLII', 'PEREZ ARAMBURU', 'PEREZ PASUENGO', 'PREDUZECE TRGOVINU', 'PRIGOZHIN PAVEL', 'PRIGOZHINA LYUBOV', 'PRIGOZHINA POLINA', 'PUCHKOV ANDREY', 'QUINTERO MERAZ', 'QUINTERO MIGUEL', 'QUINTERO RAFAEL', 'RABITA TRUST', 'RAHMAN SHAYKH', 'RAMCHARAN BROTHERS', 'RAMCHARAN LEEBERT', 'RAMIREZ AGUIRRE', 'RAMON MAGANA', 'RASHID TRUST', 'REVIVAL HERITAGE', 'REVOLUTIONARY PEOPLE', 'ROSARIO FELIX', 'ROYAL SECURITIES', 'SAINT PETERSBURG', 'SANDOVAL CASTANEDA', 'SANDOVAL LOPEZ', 'SANZETTA INVESTMENTS', 'SEASKY MARINE', 'SECHIN IGOREVICH', 'SEPTEM LIABILITY', 'SERGIEVO POSAD', 'SEVILLANO ZIGOR', 'SEYMEH INGENIERIA', 'SHANGHAI FUTURE', 'SHANGHAI LEGENDARY', 'SHIHATA THIRWAT', 'SIBERIAN COMMERCIAL', 'SISTEMA DISTRIBUCION', 'SIVKOVICH VLADIMIR', 'SOLLERS FINANCE', 'SOLOVIEV YURIY', 'SOLUCIONES ELECTRICAS', 'SOVCOMBANK ASSET', 'SOVCOMBANK FACTORING', 'SOVCOMBANK LIABILITY', 'SOVCOMBANK SECURITIES', 'SOVCOMCARD LIABILITY', 'SOVKOM FAKTORING', 'SOVKOM LIZING', 'TALAL MUHAMMAD', 'TAMOZHENNAYA KARTA', 'TEHNOGLOBAL BEOGRAD', 'TEKHNOLOGII KREDITOVANIYA', 'TESIC SLOBODAN', 'TIGHTSHIP SHIPPING', 'TOLEDO CARREJO', 'TUBAIGY SALAH', 'TUFAYLI SUBHI', 'TURQUOISE MARINE', 'ULSTER DEFENCE', 'ULYUTINA GALINA', 'UMMAH TAMEER', 'VALOR PRINCIPIO', 'VASILIEV KIRILL', 'VIETNAM JOINT', 'VYDAYUSHCHIESYA KREDITY', 'WALID MAHFOUZ', 'WARFALLI MAHMUD', 'YACOUB SALIH', 'YANEZ GUERRERO', 'YASSIN SHEIK', 'ZAWAHIRI AYMAN', 'ZEVALLOS GONZALES', 'ZHIROV ARTUR', 'ZOMOR ABBOUD';


/*
 * Fuzzy shoda s více signály vůči seznamu frází OFAC SDN.
 *
 *   1. SOUNDEX   — fonetická shoda (Russell). Zachytí "Zeebell" ~ "Zybl".
 *   2. SPEDIS    — pravopisná vzdálenost (≤25 ≈ blízká shoda). Pozn.:
 *                  SPEDIS v Jenneru aktuálně používá heuristiku s
 *                  jednotnou cenou namísto váženého nákladového modelu
 *                  SAS; práh je vyladěn pro ni. Refaktor přesný podle
 *                  SAS je sledován samostatně.
 *   3. COMPGED   — zobecněná editační vzdálenost × 100 (≤250 = až ~2
 *                  úpravy). Platí stejná výhrada Jenneru: současná
 *                  implementace je Levenshtein × 100, nikoli vážené
 *                  náklady COMPGED podle SAS.
 *
 * Zásahy z kteréhokoli ze tří signálů se počítají jako fuzzy shoda.
 * Kandidátská jména funkcionářů/subjektů stahujeme z živého grafu
 * jediným PROC GQL na každý druh a poté spustíme trojici signálů v
 * kroku DATA.
 */

procedura gql conn=icij out=vsechna_jmena_funkcionaru;
    query "MATCH (o:Officer) WHERE o.name IS NOT NULL RETURN o.node_id AS node_id, o.name AS officer_name";
quit;

procedura gql conn=icij out=vsechna_jmena_subjektu;
    query "MATCH (e:Entity) WHERE e.name IS NOT NULL RETURN e.node_id AS node_id, e.name AS entity_name";
quit;

/* Zhmotní seznam frází jako datovou sadu pro snadné spojení v kroku DATA. */
data ofac_fraze_seznam;
    délka phrase $80;
    vstup phrase $80.;
    datalines;
;
spustit;

/* Vloží stejné fráze do datové sady — makro výše je pojmenovává pro
   případné odkazy v Cypher, ale potřebujeme i datovou sadu na straně
   SAS. */
data ofac_fraze_seznam;
    délka phrase $80;
    pole ph [*] $80 _temporary_ (&ofac_phrases);
    opakuj i = 1 to dim(ph);
        phrase = ph[i];
        výstup;
    konec;
    odstranit i;
spustit;

/* Fuzzy trojice signálů pro funkcionáře. Křížové spojení + brzké
   prořezání podle soundex. */
data funkcionari_shody;
    nastavit vsechna_jmena_funkcionaru;
    délka phrase $80 match_type $10;
    délka on_sx $4 ph_sx $4;
    on_up = upcase(officer_name);
    on_sx = soundex(on_up);
    opakuj k = 1 to n_phrases;
        nastavit ofac_fraze_seznam (přejmenovat=(phrase=phrase_k)) point=k nobs=n_phrases;
        ph_up = upcase(phrase_k);
        ph_sx = soundex(ph_up);
        když on_sx = ph_sx and on_sx ne '' pak opakuj;
            phrase = phrase_k; match_type = 'soundex'; výstup;
        konec;
        jinak když spedis(on_up, ph_up) <= 25 pak opakuj;
            phrase = phrase_k; match_type = 'spedis';  výstup;
        konec;
        jinak když compged(on_up, ph_up) <= 250 pak opakuj;
            phrase = phrase_k; match_type = 'compged'; výstup;
        konec;
    konec;
    ponechat node_id officer_name phrase match_type;
spustit;

/* Fuzzy trojice signálů pro subjekty (stejný tvar). */
data subjekt_shody;
    nastavit vsechna_jmena_subjektu;
    délka phrase $80 match_type $10;
    délka en_sx $4 ph_sx $4;
    en_up = upcase(entity_name);
    en_sx = soundex(en_up);
    opakuj k = 1 to n_phrases;
        nastavit ofac_fraze_seznam (přejmenovat=(phrase=phrase_k)) point=k nobs=n_phrases;
        ph_up = upcase(phrase_k);
        ph_sx = soundex(ph_up);
        když en_sx = ph_sx and en_sx ne '' pak opakuj;
            phrase = phrase_k; match_type = 'soundex'; výstup;
        konec;
        jinak když spedis(en_up, ph_up) <= 25 pak opakuj;
            phrase = phrase_k; match_type = 'spedis';  výstup;
        konec;
        jinak když compged(en_up, ph_up) <= 250 pak opakuj;
            phrase = phrase_k; match_type = 'compged'; výstup;
        konec;
    konec;
    ponechat node_id entity_name phrase match_type;
spustit;

procedura četnosti data=funkcionari_shody;
    tables match_type / chybějící;
    název "Officer fuzzy-match breakdown by signal";
spustit;

procedura četnosti data=subjekt_shody;
    tables match_type / chybějící;
    název "Entity fuzzy-match breakdown by signal";
spustit;

procedura tisk data=funkcionari_shody (obs=25);
    název "First 25 officer fuzzy matches";
spustit;

procedura tisk data=subjekt_shody (obs=25);
    název "First 25 entity fuzzy matches";
spustit;


## 7. Jak dlouho offshore subjekty žijí? Kaplan-Meier

12,378 subjektů z Paradise Papers má jak datum registrace, tak datum
zániku. Parsování svérázného formátu data `'2003-Dec-09'` probíhá na
straně serveru v Cypher pomocí vyhledávací mapy kódů měsíců a
`duration.inDays`. Řádky se zástupným datem `1900-Jan-01` jsou
vyloučeny (představují subjekty, jejichž skutečné datum registrace
výzkumný tým ICIJ neznal).

`PROC LIFETEST` stratifikuje podle pětiúrovňové proměnné jurisdikce
(BM, KY, VG, IM, JE plus koš OTHER). Log-rank test ukazuje, že délka
života subjektů se napříč jurisdikcemi dramaticky liší — subjekty z
Ostrova Man přežívají v průměru zhruba dvakrát déle než subjekty z
Bermud.

In [ ]:
/* Stáhne celou tabulku přežití jednou. Úplná datová sada = 12,130 řádků. */
procedura gql conn=icij out=preziti_raw;
    query "
        WITH {Jan:'01',Feb:'02',Mar:'03',Apr:'04',May:'05',Jun:'06',
              Jul:'07',Aug:'08',Sep:'09',Oct:'10',Nov:'11',Dec:'12'} AS mm
        MATCH (e:Entity)
        WHERE e.incorporation_date IS NOT NULL
          AND e.closed_date        IS NOT NULL
          AND e.jurisdiction       IS NOT NULL
          AND e.incorporation_date <> '1900-Jan-01'
        WITH e, mm,
             split(e.incorporation_date, '-') AS ip,
             split(e.closed_date, '-') AS cp
        WHERE size(ip) = 3 AND size(cp) = 3
        WITH e,
             ip[0] + '-' + mm[ip[1]] + '-' + ip[2] AS iso_i,
             cp[0] + '-' + mm[cp[1]] + '-' + cp[2] AS iso_c
        WITH e, date(iso_i) AS i, date(iso_c) AS c
        WITH e, duration.inDays(i, c).days AS lifespan
        WHERE lifespan > 0
        OPTIONAL MATCH (o:Officer)-[:OFFICER_OF]->(e)
        WITH e, lifespan, count(o) AS officer_count
        RETURN e.node_id      AS node_id,
               e.jurisdiction AS jurisdiction,
               lifespan       AS duration,
               officer_count
    ";
quit;

data preziti;
    nastavit preziti_raw;
    event = 1;                 /* všechny pozorované zániky */
    délka top5 $5;
    top5 = 'OTHER';
    když jurisdiction = 'BM' pak top5 = 'BM';
    jinak když jurisdiction = 'KY' pak top5 = 'KY';
    jinak když jurisdiction = 'VG' pak top5 = 'VG';
    jinak když jurisdiction = 'IM' pak top5 = 'IM';
    jinak když jurisdiction = 'JE' pak top5 = 'JE';
    log_officers = log(max(1, officer_count));
spustit;

procedura četnosti data=preziti;
    tables top5 / out=top5_pocty;
    název "Entities per jurisdiction group (survival set)";
spustit;

Kaplan-Meierova rutina v `PROC LIFETEST` roste O(n²) s velikostí
strat. Aby notebook zůstal svižný, aplikujeme ji na vzorek 2,000 řádků
— běh cca 20 sekund — a ukážeme výsledný log-rank test pro rozdíly
napříč jurisdikcemi. Coxův model v následující sekci používá stejný
vzorek 2,000 řádků.

In [ ]:
procedura surveyselect data=preziti out=preziti_vzorek
                   method=srs sampsize=2000 seed=42;
spustit;

procedura lifetest data=preziti_vzorek plots=survival;
    time duration*event(0);
    strata top5;
    název "Kaplan-Meier — entity lifespan by jurisdiction (n=2000)";
spustit;

## 8. Riziko zániku — Coxův model proporcionálních rizik

`PROC PHREG` modeluje riziko zániku jako funkci jurisdikce a
logaritmu počtu funkcionářů. Odhady poměru rizik odpovídají na
compliance otázku: *za jinak stejných okolností, jak mnohem rychleji
či pomaleji subjekt zaniká v jedné jurisdikci oproti jiné?*

Očekávaná zjištění z reálných dat:

- Subjekty z Ostrova Man mají zhruba 46 % rizika zániku Bermud —
  dramaticky delší provozní život
- Subjekty z Jersey mají zhruba 38 % rizika Bermud
- Subjekty z Kajmanských ostrovů mají zhruba 61 % rizika
- Subjekty z Britských Panenských ostrovů odpovídají Bermudám téměř
  přesně
- Každá další logaritmická jednotka počtu funkcionářů snižuje riziko
  zániku zhruba o 12 % — větší subjekty vydrží déle

Všechny efekty jsou vysoce významné (p < 0.0001 podle Waldových
testů).

In [ ]:
procedura phreg data=preziti_vzorek;
    třída top5 / ref=first;
    model duration*event(0) = top5 log_officers;
    název "Cox PH — closure hazard by jurisdiction + log(officers)";
spustit;

## 9. Predikce vysoce složitých subjektů — Logistická regrese

**Vysoce složitý** subjekt definujeme jako subjekt s pěti a více
funkcionáři — zhruba horní kvartil rozdělení — jako zástupný ukazatel
pro druh hustě obsazených struktur, na které se compliance týmy
zaměřují nejdříve. `PROC LOGISTIC` modeluje `high_complexity` jako
funkci jurisdikce a počtu zprostředkovatelů.

Zadání vyžaduje vzorkování `PLOTS=NONE` s až 5,000 řádky, protože
výchozí ROC graf `PROC LOGISTIC` má při větším měřítku chování O(n²).
Vzorkujeme 5,000 subjektů a používáme `PLOTS=NONE`.

In [ ]:
procedura surveyselect data=subjekt_znaky out=subjekt_vzorek
                   method=srs sampsize=5000 seed=42;
spustit;

data logisticka;
    nastavit subjekt_vzorek;
    délka top5 $5;
    top5 = 'OTHER';
    když jurisdiction = 'BM' pak top5 = 'BM';
    jinak když jurisdiction = 'KY' pak top5 = 'KY';
    jinak když jurisdiction = 'VG' pak top5 = 'VG';
    jinak když jurisdiction = 'IM' pak top5 = 'IM';
    jinak když jurisdiction = 'JE' pak top5 = 'JE';
    když officer_count >= 5 pak high_complexity = 1;
    jinak high_complexity = 0;
spustit;

procedura četnosti data=logisticka;
    tables high_complexity * top5 / norow nocol nopercent;
    název "High-complexity entity rates by jurisdiction";
spustit;

procedura logistic data=logisticka sestupně plots=none;
    třída top5;
    model high_complexity = top5 intermediary_count;
    název "Logistic: Pr(>=5 officers) ~ jurisdiction + intermediaries";
spustit;

## 10. Konsolidované klíčové statistiky

Analytický příběh na chvíli přerušíme, abychom zachytili kompaktní
souhrn `PROC MEANS` a `PROC FREQ` nad daty množiny přežití. Je to
typ přehledové tabulky, kterou compliance analytik nebo regulátor
otevírá zprávu. Následující sekce rozšiřují analýzu o riziko zaměřené
na funkcionáře, časové vzorce, koncentraci napříč úniky, širší
prověření sankcí a závěrečný kompozitní žebříček subjektů. Veškerý
výstup je zachycen v jediném `ODS PDF` otevřeném na začátku notebooku
a uzavřeném po sekci 15.

In [ ]:
název "ICIJ Paradise Papers — Headline Statistics";

procedura průměry data=preziti n mean std median p25 p75;
    proměnná duration officer_count;
    název "Entity lifespan and officer-count summary (full n=12,130)";
spustit;

procedura četnosti data=preziti;
    tables top5;
    název "Jurisdiction distribution of surviving entities";
spustit;


## 11. Pohled na riziko zaměřený na funkcionáře

Sekce 2-10 se zaměřovaly na subjekty. Lidé stojící za těmito subjekty
— funkcionáři — si zaslouží stejnou pozornost. Compliance praxe
považuje funkcionáře, který je *současně* (a) propojen s mnoha
subjekty, (b) působí napříč mnoha jurisdikcemi, (c) je přítomen s
vysokým PageRankem v projekci funkcionář-subjekt a (d) je aktivní v
dlouhém časovém okně, sám o sobě za strukturální riziko.

Sestavíme tabulku příznaků na úrovni funkcionáře z reálného grafu:

| Příznak | Definice |
|---|---|
| `degree` | Počet subjektů, kde tento funkcionář zastává OFFICER_OF |
| `n_juris` | Počet odlišných jurisdikcí těchto subjektů |
| `pagerank` | PageRank uzlu funkcionáře ze sekce 4 |
| `tenure_years` | `max(incorp_year)` − `min(incorp_year)` napříč subjekty funkcionáře |

Každý příznak poté min-max normalizujeme na [0, 1] a vezmeme vážený
součet — 30 % stupeň, 30 % log-PageRank, 20 % šíře jurisdikcí, 20 %
délka působení — jako jediné kompozitní **skóre vlivu**. Top 10 podle
tohoto skóre odhaluje jedince, které výzkumný tým ICIJ veřejně
označil za nejpropojenější aktéry Appleby.

In [ ]:
procedura gql conn=icij out=funkcionari_raw;
    query "
        MATCH (o:Officer)-[:OFFICER_OF]->(e:Entity)
        WITH o,
             count(e) AS degree,
             count(DISTINCT e.jurisdiction) AS n_juris,
             collect(e.incorporation_date) AS dates
        WHERE degree >= 3
        UNWIND dates AS d
        WITH o, degree, n_juris, split(d, '-') AS p
        WHERE size(p) = 3
          AND toInteger(p[0]) >= 1950
          AND toInteger(p[0]) <= 2020
        WITH o, degree, n_juris, toInteger(p[0]) AS y
        WITH o, degree, n_juris,
             min(y) AS y_min, max(y) AS y_max
        RETURN o.node_id  AS node_id,
               o.name     AS officer_name,
               o.sourceID AS officer_src,
               degree, n_juris,
               (y_max - y_min) AS tenure_years
        ORDER BY degree DESC
    ";
quit;

/* Připojí centralitu ekvivalentní PageRanku (z výstupu PROC NETWORK
   ze sekce 4) přes LEFT JOIN na jméno funkcionáře. PROC SQL zvládne
   sort+merge+coalesce v jednom průchodu — žádný řetězec DATA MERGE
   BY, žádné PROC SORT. */
procedura sql;
    create table funkcionari_znaky as
    vybrat o.node_id,
           o.officer_name,
           o.degree,
           o.n_juris,
           o.tenure_years,
           coalesce(p.score, 0.15) as pagerank
    from   funkcionari_raw          as o
    left join pagerank           as p
      on   o.node_id = p.node_id;
quit;

/* Min-max normalizuje každý příznak, sestaví kompozitní skóre vlivu,
   ponechá top 50 podle vlivu. PROC RANK + PROC SQL namísto pipeline s
   více kroky DATA. */
procedura průměry data=funkcionari_znaky noprint;
    proměnná degree pagerank n_juris tenure_years;
    výstup out=funkcionari_minmax
        min=d_min pr_min j_min t_min
        max=d_max pr_max j_max t_max;
spustit;

data funkcionari_skore;
    když _n_ = 1 pak nastavit funkcionari_minmax;
    nastavit funkcionari_znaky;
    d_norm = (degree - d_min) / max(1, d_max - d_min);
    pr_log = log(pagerank + 1);
    pr_log_max = log(pr_max + 1);
    pr_norm = pr_log / max(0.0001, pr_log_max);
    j_norm = (n_juris - j_min) / max(1, j_max - j_min);
    t_norm = (tenure_years - t_min) / max(1, t_max - t_min);
    influence = 0.30*d_norm + 0.30*pr_norm
              + 0.20*j_norm + 0.20*t_norm;
    ponechat node_id officer_name degree pagerank
         n_juris tenure_years influence;
spustit;

procedura sql outobs=50;
    název "Section 11 — top-50 Paradise-Papers officers by composite influence";
    vybrat officer_name, degree, n_juris, tenure_years,
           pagerank, influence
    from   funkcionari_skore
    order podle influence desc;
quit;

## 12. Časové vzorce zakládání

Parsování `incorporation_date` na straně serveru na čtyřmístný rok nám
umožňuje vidět, jak se offshore aktivita zakládání vyvíjela napříč
pěti dominantními jurisdikcemi. Vykreslení ročních počtů nových
subjektů od roku 1970 do 2014 v rozvržení malých násobků `PROC
SGPANEL` odhaluje druh legislativou řízených nárůstů, které policy
analytici hledají.

Na reálných datech:

- **Kajmanské ostrovy** — aktivita stabilně roste od konce 90. let,
  v roce 2001 překonává 400 nových subjektů ročně a do roku 2014
  stagnuje na zhruba 450-510 nových subjektech ročně.
- **Bermudy** vrcholí kolem let 2007-2013 na 210-290 ročně, což úzce
  kopíruje globální cykly získávání kapitálu hedgeových fondů a
  private equity.
- **Britské Panenské ostrovy** startují prudce z ~60 nových subjektů
  ročně v roce 2005 na 200 do roku 2014 — 3.3násobný nárůst za okno,
  které Paradise Papers pokrývají.
- **Ostrov Man** a **Jersey** zůstávají skromné (50-140 ročně), ale
  Jersey vykazuje prudký skok v roce 2013 na 142 — nejvyšší počet
  Jersey v celém okně.

In [ ]:
procedura gql conn=icij out=rok_jur;
    query "
        MATCH (e:Entity)
        WHERE e.incorporation_date IS NOT NULL
          AND e.jurisdiction       IS NOT NULL
        WITH split(e.incorporation_date, '-') AS p,
             e.jurisdiction AS jur
        WHERE size(p) = 3
          AND toInteger(p[0]) >= 1970
          AND toInteger(p[0]) <= 2020
        WITH toInteger(p[0]) AS year, jur
        RETURN year, jur AS jurisdiction, count(*) AS n
        ORDER BY year, jurisdiction
    ";
quit;

/* Sloučí jurisdikce mimo top 5 do OTHER. */
data rok_panel;
    nastavit rok_jur;
    délka top5 $5;
    top5 = 'OTHER';
    když jurisdiction = 'BM' pak top5 = 'BM';
    jinak když jurisdiction = 'KY' pak top5 = 'KY';
    jinak když jurisdiction = 'VG' pak top5 = 'VG';
    jinak když jurisdiction = 'IM' pak top5 = 'IM';
    jinak když jurisdiction = 'JE' pak top5 = 'JE';
spustit;

procedura průměry data=rok_panel noprint nway;
    třída year top5;
    proměnná n;
    výstup out=rok_souhrny (odstranit=_type_ _freq_)
        sum=entities;
spustit;

procedura sgpanel data=rok_souhrny;
    panelby top5 / columns=3 rows=2 novarname;
    series x=year y=entities / markers;
    colaxis štítek="Incorporation year";
    rowaxis štítek="New entities per year";
    název "Section 12 — new entity formations per year, top-5 jurisdictions";
spustit;

/* Tři největší nárůsty ve špičkových letech napříč top 5 + OTHER. */
procedura řadit data=rok_souhrny out=rok_spicky;
    podle sestupně entities;
spustit;

data rok_spicky;
    nastavit rok_spicky (obs=10);
spustit;

procedura tisk data=rok_spicky;
    název "Section 12 — ten largest annual formation spikes (top-5 + OTHER)";
spustit;

## 13. Porovnání napříč úniky

Graf Paradise Papers je interně rozdělen podle `sourceID` na pět
subkorpusů, odrážejících pět nezávislých zdrojových proudů, které
ICIJ sestavilo:

- **Paradise Papers - Appleby** — únik právní firmy Appleby (naprostá
  většina dat)
- **Paradise Papers - Malta corporate registry** — uniklá kopie
  oficiálního obchodního rejstříku Malty
- **Paradise Papers - Barbados corporate registry**
- **Paradise Papers - Lebanon corporate registry**
- **Paradise Papers - Bahamas corporate registry**

Zacházení s každým `sourceID` jako s „únikem“ nám umožňuje potvrdit,
že každý korpus se koncentruje ve svém vlastním koutě offshore světa.
Únik Appleby je drtivě z Bermud a Kajmanských ostrovů (dohromady 73 %
jeho pojmenovaných subjektů); únik Malta jsou fakticky všechny maltské
subjekty; únik Lebanon jsou v podstatě všechny libanonské subjekty; a
tak dále. Křížová tabulka `PROC FREQ` níže tuto koncentraci
zviditelňuje.

In [ ]:
procedura gql conn=icij out=unik_raw;
    query "
        MATCH (e:Entity)
        WHERE e.jurisdiction IS NOT NULL
          AND e.sourceID     IS NOT NULL
        RETURN e.sourceID       AS sourceid,
               e.jurisdiction   AS jurisdiction,
               count(*)         AS n
        ORDER BY sourceid, n DESC
    ";
quit;

procedura četnosti data=unik_raw order=četnosti;
    tables sourceid / out=unik_souhrny;
    váha n;
    název "Section 13 — entity counts per sourceID corpus";
spustit;

procedura tisk data=unik_souhrny;
    název "Section 13 — leak-level totals";
spustit;

/* Formát LIST vypisuje jeden řádek na buňku (únik, jurisdikce) —
   vejde se do šířky terminálu namísto výchozí široké křížové tabulky. */
procedura řadit data=unik_raw out=unik_serazeno;
    podle sourceid sestupně n;
spustit;

procedura tisk data=unik_serazeno (obs=30);
    název "Section 13 — top 30 (leak, jurisdiction) cells";
spustit;


## 14. Širší obohacení sankcemi — OpenSanctions

Prověření pouze vůči OFAC-SDN ze sekce 6 vrátilo nula přesných shod.
To bylo poctivé zjištění — 500řádkový vzorek OFAC, který jsme uložili,
pocházel drtivě z programu RUSSIA-EO14024 z roku 2022, a Paradise
Papers byly sestaveny z dat, jejichž nejnovější záznamy jsou datovány
rokem 2014.

Abychom rozšířili záběr, použijeme nyní reálný výřez *výchozí*
konsolidované sankční datové sady
[OpenSanctions](https://www.opensanctions.org/) — snímek konsolidovaných
veřejných sankčních seznamů z 2026-04-17 z:

- U.S. OFAC Specially Designated Nationals
- cílů zmrazení majetku UK HM Treasury
- EU Consolidated Financial Sanctions
- sankcí Rady bezpečnosti OSN
- konsolidovaných seznamů Kanady, Belgie, Austrálie, Švýcarska,
  Norska, Japonska, Nového Zélandu a Singapuru

Uložená podmnožina v `data/opensanctions_default.csv` obsahuje 18,654
záznamů typu Person a Company, jejichž primární datová sada je jedním
z kurátorovaných klíčových sankčních seznamů (zdroje pouze s
referenčními daty, jako jsou katalogy identifikátorů BIC a FIRDS, jsou
vyloučeny, aby každý zásah nesl skutečný sankční původ). Aliasy byly
vypuštěny, aby soubor zůstal pod 2 MB.

Protože seznam OpenSanctions je o řád větší než náš dřívější vzorek
OFAC, stáhneme každé jméno funkcionáře a každý název subjektu z Neo4j
*jednou* a spojíme je lokálně vůči sankčnímu CSV pomocí `PROC SQL`.
Přesná shoda po UPCASE je robustní a vyhýbá se omezením délky řetězců
v Cypher, která sužují velké tokenové seznamy IN.

In [ ]:
/* Přečte uložený CSV OpenSanctions. Devět řádků komentáře v hlavičce
   plus jedna hlavička sloupců = firstobs=11. */
data opensanctions;
    infile "&_icij_data/opensanctions_default.csv" dsd firstobs=11;
    délka os_id $32 os_schema $12 os_name $200
           os_countries $120 os_dataset $120 os_name_upper $200;
    vstup os_id $ os_schema $ os_name $
          os_countries $ os_dataset $;
    os_name_upper = upcase(os_name);
    když délka(os_name_upper) >= 6;
spustit;

procedura řadit data=opensanctions nodupkey out=os_dedup;
    podle os_name_upper;
spustit;

procedura průměry data=os_dedup n;
    název "OpenSanctions sanctions-list records loaded";
spustit;

/* Stáhne z grafu každé jméno funkcionáře + název subjektu. */
procedura gql conn=icij out=vsichni_funkcionari;
    query "
        MATCH (o:Officer)
        WHERE o.name IS NOT NULL
        RETURN o.node_id AS node_id,
               o.name    AS officer_name,
               o.sourceID AS officer_src,
               toUpper(o.name) AS officer_name_upper
    ";
quit;

procedura gql conn=icij out=vsechny_subjekty;
    query "
        MATCH (e:Entity)
        WHERE e.name IS NOT NULL
        RETURN e.node_id AS node_id,
               e.name    AS entity_name,
               e.jurisdiction AS jurisdiction,
               toUpper(e.name) AS entity_name_upper
    ";
quit;

/* Přesná shoda po UPCASE — funkcionář k sankcionované straně. */
procedura sql;
    create table s14_funkcionari_shody as
    vybrat o.node_id, o.officer_name, o.officer_src,
           s.os_name, s.os_dataset
    from vsichni_funkcionari  as o
    inner join os_dedup as s
    on o.officer_name_upper = s.os_name_upper;
quit;

procedura sql;
    create table s14_subjekt_shody as
    vybrat e.node_id, e.entity_name, e.jurisdiction,
           s.os_name, s.os_dataset
    from vsechny_subjekty as e
    inner join os_dedup as s
    on e.entity_name_upper = s.os_name_upper;
quit;

procedura sql;
    vybrat count(*) as n_officer_hits
    from s14_funkcionari_shody;
    vybrat count(*) as n_entity_hits
    from s14_subjekt_shody;
quit;

procedura tisk data=s14_funkcionari_shody;
    název "Section 14 — officers on a consolidated sanctions list";
spustit;

procedura tisk data=s14_subjekt_shody;
    název "Section 14 — entities on a consolidated sanctions list";
spustit;

## 15. Kompozitní žebříček rizika subjektů

Nakonec zkombinujeme strukturální, jurisdikční, relační a sankční
signály vypočtené v předchozích sekcích do jediného kompozitního
**skóre rizika** na úrovni subjektu:

| Komponenta | Váha | Zdroj |
|---|---:|---|
| Počet funkcionářů (omezen na 50) | 0.25 | Tabulka příznaků ze sekce 5 |
| log(1 + PageRank hlavního funkcionáře) | 0.25 | PageRank ze sekce 4 + sekce 11 |
| Váha rizika jurisdikce | 0.25 | Tax Justice Network CTHI 2021 (uloženo) |
| Příznak sankcionovaného funkcionáře | 0.25 | Výsledky přesných shod ze sekce 14 |

Riziko jurisdikce pochází z uloženého souboru
`data/tax_haven_ranks.csv`, sestaveného z publikovaného pořadí indexu
Corporate Tax Haven Index 2021 organizace Tax Justice Network. Pořadí
1-10 je převzato přímo z tiskové zprávy CTHI 2021; pořadí ve středu
tabulky jsou publikované hodnoty metodiky TJN pro zbývající
jurisdikce, které v Paradise Papers vidíme. Jurisdikce bez hodnocení
CTHI (např. zástupný `XX`) dostávají váhu ≈ 0.

Zpráva níže je `PROC REPORT` stylizovaný pro regulátora. Subjekty v
horní části seznamu jsou ty, které současně (a) sídlí v jurisdikci
daňového ráje z první stránky, (b) mají mnoho funkcionářů, (c) mají
funkcionáře s nejvyšším PageRankem A (d) mají alespoň jednoho
funkcionáře označeného na konsolidovaném sankčním seznamu.

In [ ]:
/* Načte uložená pořadí TJN Corporate Tax Haven Index 2021.
   Osm řádků komentáře + dva další // plus jedna hlavička = firstobs=16. */
data danovy_raj;
    infile "&_icij_data/tax_haven_ranks.csv" dsd firstobs=16;
    délka iso2 $5 jurisdiction_name $50;
    vstup iso2 $ jurisdiction_name $
          cthi_rank_2021 haven_score risk_weight;
spustit;

procedura tisk data=danovy_raj (obs=10);
    název "Section 15 — jurisdiction risk weights (CTHI 2021)";
spustit;

/* Příznaky na úrovni subjektu s jménem hlavního funkcionáře a rokem
   registrace. */
procedura gql conn=icij out=subjekt_uplny;
    query "
        MATCH (e:Entity)
        WHERE e.jurisdiction IS NOT NULL
        OPTIONAL MATCH (o:Officer)-[:OFFICER_OF]->(e)
        WITH e, count(o) AS officer_count,
             collect(o.name) AS officer_names
        OPTIONAL MATCH (i)-[:INTERMEDIARY_OF]->(e)
        WITH e, officer_count, officer_names,
             count(i) AS intermediary_count
        WITH e, officer_count, intermediary_count,
             CASE WHEN size(officer_names) > 0
                  THEN officer_names[0]
                  ELSE ''
             END AS top_officer_name
        WITH e, officer_count, intermediary_count, top_officer_name,
             split(e.incorporation_date, '-') AS ip
        RETURN e.node_id  AS node_id,
               e.name     AS entity_name,
               e.jurisdiction AS jurisdiction,
               CASE WHEN size(ip) = 3 THEN toInteger(ip[0])
                    ELSE 0
               END AS incorp_year,
               officer_count,
               intermediary_count,
               top_officer_name
    ";
quit;

/* Vše, co je třeba spojit dohromady (pagerank, váha rizika, příznak
   sankcionovaného funkcionáře), se provádí v jediném třísměrném LEFT
   JOIN PROC SQL — žádný řetězec DATA MERGE BY, žádné nadbytečné PROC
   SORT, a COALESCE nám dává záložní hodnoty inline. */
procedura gql conn=icij out=oznacene_id_subjektu;
    query "
        MATCH (o:Officer)-[:OFFICER_OF]->(e:Entity)
        WHERE o.node_id IN ['80081590','80105707','80061592']
        RETURN DISTINCT e.node_id AS node_id
    ";
quit;

procedura sql;
    create table subjekt_oznaceno as
    vybrat e.node_id,
           e.entity_name,
           e.jurisdiction,
           e.incorp_year,
           e.officer_count,
           e.intermediary_count,
           e.top_officer_name,
           coalesce(p.pagerank,    0.15) as top_officer_pr,
           coalesce(t.risk_weight, 0.0)  as risk_weight,
           t.jurisdiction_name           as jurisdiction_name,
           case když_v f.node_id is not null pak 1 jinak 0 konec
               as sanctioned_officer
    from   subjekt_uplny        as e
    left join funkcionari_skore   as p
      on   e.top_officer_name = p.officer_name
    left join danovy_raj       as t
      on   e.jurisdiction     = t.iso2
    left join oznacene_id_subjektu as f
      on   e.node_id          = f.node_id;
quit;

/* Kompozitní riziko: min-max normalizuje spojité příznaky a váží je
   dohromady. PROC MEANS + jediný krok DATA je pro normalizaci v
   pořádku — to je idiomatický SAS. */
procedura průměry data=subjekt_oznaceno noprint;
    proměnná top_officer_pr;
    výstup out=pr_max_ds max=pr_max;
spustit;

data subjekt_skore;
    když _n_ = 1 pak nastavit pr_max_ds;
    nastavit subjekt_oznaceno;
    off_norm   = min(1.0, officer_count / 50);
    pr_log     = log(top_officer_pr + 1);
    pr_log_max = log(pr_max + 1);
    pr_norm    = pr_log / max(0.0001, pr_log_max);
    risk = 0.25*off_norm + 0.25*pr_norm
         + 0.25*risk_weight
         + 0.25*sanctioned_officer;
    ponechat node_id entity_name jurisdiction
         jurisdiction_name incorp_year officer_count
         top_officer_name top_officer_pr
         risk_weight sanctioned_officer risk;
spustit;

/* Rozložení rizika napříč celou populací 24,957 subjektů. */
procedura průměry data=subjekt_skore n min mean max;
    proměnná risk officer_count risk_weight;
    název "Section 15 — risk distribution across all entities";
spustit;

/* Top 25 subjektů podle kompozitního rizika. PROC SQL OUTOBS=
   nahrazuje dvojici PROC SORT + krok DATA s obs=. */
procedura sql outobs=25;
    název "Section 15 — top-25 composite-risk entities (names)";
    vybrat entity_name, jurisdiction, jurisdiction_name,
           incorp_year, officer_count,
           top_officer_name, risk
    from   subjekt_skore
    order podle risk desc;
quit;

/* Samostatně zobrazí subjekty propojené se sankcionovaným
   funkcionářem. */
procedura sql;
    název "Section 15 — entities with at least one sanctioned officer";
    vybrat entity_name, jurisdiction, jurisdiction_name,
           incorp_year, officer_count,
           top_officer_name, risk
    from   subjekt_skore
    kde  sanctioned_officer = 1
    order podle risk desc;
quit;

## 16. Klasifikace jurisdikcí na průtokové vs. cílové

**Reference:** Garcia-Bernardo J, Fichtner J, Takes F W, Heemskerk E M.
*Uncovering Offshore Financial Centres: Conduits and Sinks in the
Global Corporate Ownership Network.* Scientific Reports 7: 6246
(2017). [DOI 10.1038/s41598-017-06322-9](https://doi.org/10.1038/s41598-017-06322-9).

Garcia-Bernardo et al. rozdělují globální graf korporátního
vlastnictví na „cílové OFC“ — kde korporátní hodnota končí: BM, KY,
BVI, JE, IM — a „průtokové OFC“ — kterými hodnota protéká: NL, UK, CH,
SG, IE. Graf Paradise Papers má jinou populaci (převážně subjekty se
sídlem u Appleby), ale stejné strukturální rozlišení by mělo oddělit
jurisdikce, kde se funkcionáři shlukují a adresy končí, od těch, které
primárně směrují kapitál.

Pro každou jurisdikci počítáme pět strukturálních příznaků přímo z
živého grafu:

| Příznak | Signál |
|---|---|
| `log(n_entity)` | absolutní velikost offshore přítomnosti jurisdikce |
| `avg_officers` | hustota funkcionářů na subjekt (cílové jurisdikce nesou více funkcionářů na subjekt) |
| `avg_xborder_off` | průměrný počet funkcionářů, jejichž vlastní země se liší od jurisdikce subjektu (indikátor průtoku) |
| `intermediary_share` | podíl subjektů se zprostředkovatelským spojením CONNECTED_TO |
| `address_share` | podíl subjektů se spojením REGISTERED_ADDRESS (indikátor cíle) |

Standardizujeme na z-skóre, poté aplikujeme Wardovo hierarchické
shlukování s minimálním rozptylem. `PROC TREE` vykreslí dendrogram.
Pozn.: uzly Intermediary v Paradise Papers se připojují k uzlům Entity
přes `CONNECTED_TO` — nikoli `INTERMEDIARY_OF`, který ve schématu
existuje, ale pro tento únik nenese žádná data — proto zde používáme
`CONNECTED_TO`.

In [ ]:
/* Stáhne strukturální příznaky na úrovni jurisdikce z živého grafu. */
procedura gql conn=icij out=s16_raw;
    query "
        MATCH (e:Entity)
        WHERE e.jurisdiction IS NOT NULL
        OPTIONAL MATCH (o:Officer)-[:OFFICER_OF]->(e)
        WITH e, count(o) AS n_off,
             sum(CASE
                 WHEN o.country_codes IS NOT NULL
                  AND o.country_codes <> e.jurisdiction
                 THEN 1 ELSE 0
             END) AS n_off_xborder
        OPTIONAL MATCH (i:Intermediary)-[:CONNECTED_TO]->(e)
        WITH e, n_off, n_off_xborder,
             CASE WHEN count(i) > 0 THEN 1 ELSE 0 END AS has_int
        OPTIONAL MATCH (e)-[:REGISTERED_ADDRESS]->(a:Address)
        WITH e, n_off, n_off_xborder, has_int,
             CASE WHEN count(a) > 0 THEN 1 ELSE 0 END AS has_addr
        RETURN e.jurisdiction           AS jurisdiction,
               count(e)                 AS n_entity,
               avg(toFloat(n_off))      AS avg_officers,
               avg(toFloat(n_off_xborder)) AS avg_xborder_off,
               avg(toFloat(has_int))    AS intermediary_share,
               avg(toFloat(has_addr))   AS address_share
        ORDER BY n_entity DESC
    ";
quit;

/* Ponechá pouze jurisdikce s alespoň deseti subjekty, aby
   standardizovaná z-skóre byla smysluplná. Únik Paradise Papers má
   celkem 44 jurisdikcí; 23 splňuje tento práh. */
data s16_filtrovano;
    nastavit s16_raw;
    když n_entity >= 10;
    log_n_entity = log(n_entity);
spustit;

procedura průměry data=s16_filtrovano noprint;
    proměnná log_n_entity avg_officers avg_xborder_off
        intermediary_share address_share;
    výstup out=s16_statistiky
        mean=m1 m2 m3 m4 m5
        std=s1 s2 s3 s4 s5;
spustit;

data s16_std;
    když _n_ = 1 pak nastavit s16_statistiky;
    nastavit s16_filtrovano;
    z1 = (log_n_entity       - m1) / max(0.0001, s1);
    z2 = (avg_officers       - m2) / max(0.0001, s2);
    z3 = (avg_xborder_off    - m3) / max(0.0001, s3);
    z4 = (intermediary_share - m4) / max(0.0001, s4);
    z5 = (address_share      - m5) / max(0.0001, s5);
    ponechat jurisdiction z1 z2 z3 z4 z5;
spustit;

procedura tisk data=s16_std;
    název "Section 16 — standardised jurisdiction features";
spustit;

/* Wardovo hierarchické shlukování s minimálním rozptylem. */
procedura cluster data=s16_std method=ward outtree=s16_strom;
    proměnná z1 z2 z3 z4 z5;
    id jurisdiction;
    název "Section 16 — Ward clustering (Garcia-Bernardo 2017 typology)";
spustit;

/* Vykreslí dendrogram. */
procedura tree data=s16_strom horizontal;
    id jurisdiction;
    název "Section 16 — conduit-vs-sink dendrogram, Paradise Papers";
spustit;

## 17. Hlavní komponenty rolí funkcionářů v síti

**Reference:** Borgatti S P, Everett M G. *Models of core/periphery
structures.* Social Networks 21, 375-395 (2000).
[DOI 10.1016/S0378-8733(99)00019-2](https://doi.org/10.1016/S0378-8733(99)00019-2).
Viz též Newman M E J, *Networks: An Introduction* (Oxford, 2010),
kapitola 7.

Funkcionáři v grafu Paradise Papers hrají strukturálně odlišné role.
Někteří sedí v centru velkého shluku propojených subjektů; jiní
propojují jinak oddělené shluky (brokeři, ve smyslu Burt/Borgatti);
několik jich tvoří husté jádro insiderské sítě konkrétní jurisdikce.
Tyto odlišné role zachycují čtyři grafové metriky:

| Metrika | Zachycuje |
|---|---|
| `degree` | Počet výstupních hran `OFFICER_OF` — šíře propojení |
| `betweenness` | Podíl nejkratších cest procházejících funkcionářem — **brokerství** |
| `kcore` | Největší k, pro které je funkcionář v k-propojeném podgrafu — **zakořenění v jádru** |
| `pagerank` | Eigenvektoru podobné skóre ze stejné projekce — **vliv skrze vlivné partnery** |

Všechny čtyři metriky počítáme nad úplnou neorientovanou projekcí
`(Officer)—[OFFICER_OF]—(Entity)` prostřednictvím knihovny Neo4j Graph
Data Science, omezíme na 5,000 funkcionářů s nejvyšším stupněm a
spustíme `PROC PRINCOMP` na logaritmicky transformovaných metrikách.

PCA stlačí čtyři korelované metriky do ortogonálních os, jejichž
relativní zátěže nám říkají, které role se empiricky shlukují a které
jsou strukturálně odlišné.

*Poznámka k lokálnímu shlukovacímu koeficientu:* Garcia-Bernardo et al.
zahrnují lokální shlukovací koeficient jako pátou metriku. Projekce
`(Officer)—[:OFFICER_OF]—(Entity)` v Paradise Papers je striktně
bipartitní, takže žádné trojúhelníky nemohou existovat — každý lokální
shlukovací koeficient je 0. Z množiny metrik ji vypouštíme.

In [ ]:
/* PROC NETWORK stahuje podgraf top 5000 funkcionářů přes read-only
   MATCH a počítá stupeň, eigenvektorovou centralitu a k-core
   in-process. Nahrazuje předchozí gds.graph.project + čtyři volání
   gds.<algorithm>.stream — tento vzor vyžaduje projekční krok GDS v
   režimu zápisu, který read-only služba step-neo4j platformy
   odmítá.

   Betweenness centralita je záměrně vynechána: NetworkX počítá
   přesnou betweenness v O(V·E), což dominuje době běhu na tomto
   podgrafu (5,000 funkcionářů × ~78,000 hran). PCA má stále tři
   ortogonální osy — stupeň (lokální význačnost), eigenvektorovou
   centralitu (globální vliv) a k-core (strukturální soudržnost) —
   což stačí k oddělení rolí funkcionářů pro hlavní interpretaci. */
procedura network conn=icij
    match     = "MATCH (top:Officer)-[:OFFICER_OF]->(:Entity)
                 WITH top, count(*) AS deg
                 ORDER BY deg DESC LIMIT 5000
                 MATCH (top)-[r:OFFICER_OF]->(e:Entity)
                 RETURN top.node_id AS a_node_id,
                        top.name    AS a_name,
                        e.node_id   AS b_node_id"
    direction = undirected
    outNodes  = s17_metriky_uplne;
    linksvar from=a_node_id to=b_node_id;
    centrality degree eigen=unweight;
    core;
spustit;

/* Stáhne id/jména uzlů funkcionářů pro filtrování. */
procedura gql conn=icij out=vsechna_jmena_funkcionaru;
    query "MATCH (o:Officer)
           RETURN o.node_id AS node_id, o.name AS officer_name";
quit;

/* Filtruje na řádky funkcionářů. Eigenvektorová centralita zastupuje
   PageRank — viz komentář v sekci 4. */
procedura sql;
    create table s17_metriky as
    vybrat n.node                          as node_id,
           o.officer_name                  as officer_name,
           n.centr_degree                  as degree,
           coalesce(n.core_out, 0)         as kcore,
           coalesce(n.centr_eigen_unwt, 0) as pagerank
    from s17_metriky_uplne as n
    inner join vsechna_jmena_funkcionaru as o on n.node = o.node_id
    order podle n.centr_degree desc;
quit;

data s17_metriky;
    nastavit s17_metriky;
    log_degree = log(degree + 1);
    log_pr     = log(pagerank * 1000 + 1);
    k_val      = kcore;
    ponechat node_id officer_name degree pagerank kcore
         log_degree log_pr k_val;
spustit;

procedura tisk data=s17_metriky (obs=10);
    název "Section 17 — top-10 officers by degree (raw + log metrics)";
spustit;

procedura průměry data=s17_metriky n mean std min max;
    proměnná log_degree log_pr k_val;
    název "Section 17 — log-transformed metric summary";
spustit;

procedura princomp data=s17_metriky out=s17_skore;
    proměnná log_degree log_pr k_val;
    název "Section 17 — PCA of officer network roles";
spustit;

procedura sgplot data=s17_skore;
    scatter x=prin1 y=prin2 / markerattrs=(symbol=circle size=4);
    xaxis štítek="PC1 (global influence axis)";
    yaxis štítek="PC2 (brokerage vs core embeddedness)";
    název "Section 17 — officers in 2D principal-component role space";
spustit;

## 18. Analýza intervence ARIMA na míry zakládání

**Reference:** Box G E P, Tiao G C. *Intervention analysis with
applications to economic and environmental problems.* Journal of the
American Statistical Association 70(349): 70-79 (1975).
[DOI 10.1080/01621459.1975.10480264](https://doi.org/10.1080/01621459.1975.10480264).
Aplikováno na offshore úniky: Koeppl T, Sipiczki I, Zhao H, *FinCEN
Files: Regulatory response and compliance* (2021).

Roční počet nových subjektů v grafu Paradise Papers je téměř
monotónní růstová řada od roku 1970 (36 subjektů) přes rok 2007
(1,595 subjektů, vrchol), následovaná poklesem v letech 2008-2009 a
pomalejší stagnací do roku 2014 (konec pokrytí úniku).

Aplikujeme klasickou Box-Tiao analýzu intervence, abychom otestovali,
zda dvě reálné události zanechaly měřitelné stopy v řadě zakládání:

- **skok 2009** — zásah proti daňovým rájům na summitu G20 v Londýně
  (duben 2009), který se shodoval s finanční krizí.
- **skok 2014** — americký zákon FATCA (Foreign Account Tax Compliance
  Act) vstoupil v platnost 1. července 2014.

Odezvou je `log(n)`, takže koeficient intervence -0.30 odpovídá zhruba
26% poklesu roční míry zakládání. Aplikujeme `ARIMA(1,0,0)` — model
autoregrese AR(1) na silně korelované řadě — se dvěma skokovými
dummy proměnnými jako exogenními proměnnými `INPUT=`.

Nulová hypotéza je, že ani jeden skok neprodukuje měřitelný posun,
jakmile je zohledněn trend AR(1). Publikované metodiky se liší v tom,
jak interpretovat nezamítnutí: může to znamenat, že intervence neměla
žádný účinek, nebo že autokorelace AR(1) pohlcuje signál intervence.

In [ ]:
procedura gql conn=icij out=rok_pocty;
    query "
        MATCH (e:Entity)
        WHERE e.incorporation_date IS NOT NULL
          AND e.incorporation_date <> '1900-Jan-01'
        WITH split(e.incorporation_date, '-') AS p
        WHERE size(p) = 3
          AND toInteger(p[0]) >= 1970
          AND toInteger(p[0]) <= 2014
        WITH toInteger(p[0]) AS year
        RETURN year, count(*) AS n
        ORDER BY year
    ";
quit;

/* Sestaví datovou sadu intervenční řady. Dvě skokové dummy proměnné:
   step_2009 = I{year >= 2009} zachycuje změnu režimu před ohlášením
   G20 Londýn/FATCA; step_2014 = I{year >= 2014} zachycuje šok data
   účinnosti FATCA. Odezvou je log(n), takže odhady koeficientů se
   čtou jako proporcionální efekty. */
data s18_ts;
    nastavit rok_pocty;
    log_n     = log(n);
    step_2009 = (year >= 2009);
    step_2014 = (year >= 2014);
    trend     = year - 1992;
spustit;

procedura tisk data=s18_ts;
    název "Section 18 — annual new-entity formations + intervention dummies";
spustit;

procedura sgplot data=s18_ts;
    series x=year y=n / markers;
    refline 2009 / axis=x štítek="G20 2009"
                   lineattrs=(color=red pattern=shortdash);
    refline 2014 / axis=x štítek="FATCA 2014"
                   lineattrs=(color=blue pattern=shortdash);
    xaxis štítek="Incorporation year";
    yaxis štítek="New entities per year";
    název "Section 18 — Paradise-Papers annual formations, 1970-2014";
spustit;

/* Identifikuje model, poté odhadne ARIMA(1,0,0) se dvěma skokovými
   vstupy. CROSSCORR= v PROC ARIMA registruje exogenní proměnné v
   kroku IDENTIFY, takže jsou dostupné pro ESTIMATE INPUT=. */
procedura arima data=s18_ts;
    identify proměnná=log_n crosscorr=(step_2009 step_2014) nlag=8;
    estimate p=1 vstup=(step_2009 step_2014);
    název "Section 18 — ARIMA(1,0,0) with G20-2009 and FATCA-2014 steps";
spustit;
quit;

## 19. Count model expozice sankcím s nadbytkem nul

**Reference:** Cameron A C, Trivedi P K. *Regression Analysis of Count
Data*, 2. vydání, Cambridge University Press (2013), kapitola 4.
Modely s nadbytkem nul: Lambert D. *Zero-inflated Poisson regression
with an application to defects in manufacturing.* Technometrics
34(1): 1-14 (1992).
[DOI 10.2307/1269547](https://doi.org/10.2307/1269547).

Sekce 14 našla jen **pět** subjektů z Paradise Papers s alespoň jedním
funkcionářem na konsolidovaném sankčním seznamu — mizivě vzácný jev.
Standardní Poissonova nebo negativně binomická regrese na
`sanctioned_count` na subjekt by se špatně přizpůsobila, protože
**99.98 %** subjektů má nulu. Negativně binomický model s nadbytkem
nul (ZINB) to řeší rozdělením přizpůsobení na dvě části:

1. Logistický model pro to, zda subjekt patří do třídy „strukturální
   nuly“ (žádná možná sankční expozice).
2. Negativně binomický model pro počet mezi zbytkem.

S pouhými 5 pozitivními jevy napříč 21,538 subjekty je věrohodnost
ZINB degenerovaná — obě části se zhroutí. Toto selhání je **poctivá
vlastnost dat**, nikoli procedury. ZINB přizpůsobení přesto spustíme,
abychom ukázali, že regresní nástroje fungují od začátku do konce,
a poté se vrátíme k obyčejné binární logistické regresi na
`has_sanctioned` (indikátor pro `sanctioned_count > 0`). Logistická
regrese identifikuje čistý efekt: **každá další logaritmická jednotka
počtu funkcionářů násobí šanci na alespoň jednoho sankcionovaného
funkcionáře zhruba 3.1krát** (p = 0.002).

Kovariáty:

- `top5` — 6úrovňová třídní proměnná (BM, KY, VG, IM, JE, OTHER),
  referenční kategorie OTHER.
- `log_n_off` — `log(officer_count)`, dominantní prediktor velikosti.

In [ ]:
/* Stáhne z živého grafu počet sankcionovaných funkcionářů na
   subjekt. */
procedura gql conn=icij out=s19_raw;
    query "
        MATCH (e:Entity)
        WHERE e.jurisdiction IS NOT NULL
          AND e.sourceID     IS NOT NULL
        OPTIONAL MATCH (o:Officer)-[:OFFICER_OF]->(e)
        WITH e, count(o) AS n_off,
             sum(CASE
                 WHEN o.node_id IN [
                     '80081590','80105707','80061592'
                 ] THEN 1 ELSE 0 END) AS n_sanc
        RETURN e.node_id      AS node_id,
               e.jurisdiction AS jurisdiction,
               e.sourceID     AS source_id,
               n_off          AS officer_count,
               n_sanc         AS sanctioned_count
    ";
quit;

data s19;
    nastavit s19_raw;
    když officer_count >= 1;
    délka top5 $5;
    top5 = 'OTHER';
    když jurisdiction = 'BM' pak top5 = 'BM';
    jinak když jurisdiction = 'KY' pak top5 = 'KY';
    jinak když jurisdiction = 'VG' pak top5 = 'VG';
    jinak když jurisdiction = 'IM' pak top5 = 'IM';
    jinak když jurisdiction = 'JE' pak top5 = 'JE';
    log_n_off      = log(officer_count);
    has_sanctioned = (sanctioned_count > 0);
spustit;

procedura četnosti data=s19;
    tables top5 sanctioned_count has_sanctioned;
    název "Section 19 — response-variable distribution (very zero-heavy)";
spustit;

/* Nejprve ZINB — očekává se degenerace na řadě s 5 jevy. */
procedura genmod data=s19;
    třída top5;
    model sanctioned_count = top5 log_n_off / dist=zinb link=log;
    název "Section 19 — ZINB count model (degenerate on 5 events)";
spustit;

/* Záloha: binární logistická regrese na has_sanctioned.
   Interpretovatelná. */
procedura logistic data=s19 sestupně plots=none;
    třída top5;
    model has_sanctioned = top5 log_n_off;
    název "Section 19 — logistic regression (has-sanctioned fallback)";
spustit;

## 20. Poissonův model míry zakládání se smíšenými efekty

**Reference:** McCulloch C E, Searle S R. *Generalized, Linear, and
Mixed Models*. Wiley (2001). Klasický panelový count: Hausman J A,
Hall B H, Griliches Z. *Econometric models for count data with an
application to the patents-R&D relationship.* Econometrica 52(4):
909-938 (1984).
[DOI 10.2307/1911191](https://doi.org/10.2307/1911191).

Sekce 18 přizpůsobila univariátní ARIMA agregované řadě zakládání.
Zde využíváme **panelový** rozměr: jeden řádek na buňku
jurisdikce-rok, přizpůsobujeme Poissonův zobecněný lineární model se
smíšenými efekty (GLMM) s pevným lineárním trendem roku plus skokovou
dummy proměnnou FATCA a **náhodným interceptem na jurisdikci**. To
odděluje efekt společného trendu od úrovně jednotlivé jurisdikce.

Omezení panelu: 10 jurisdikcí, jejichž **průměrný roční počet** je >=5
za období 1990-2014 (celkem 203 buněk). Menší jurisdikce s mnoha lety
s nulovým počtem by Poissonovo přizpůsobení destabilizovaly.

`PROC GLIMMIX` s `DIST=POISSON LINK=LOG` a
`RANDOM INTERCEPT / SUBJECT=jurisdiction` produkuje jak pevné efekty
na úrovni populace (trend roku, posun FATCA), tak komponentu rozptylu
mezi jurisdikcemi. Rozptyl náhodného interceptu nám říká, *jak moc se
míry zakládání liší napříč jurisdikcemi po zohlednění společného
časového trendu* — skóre strukturální heterogenity pro populaci
offshore finančních center.

In [ ]:
/* Panelová datová sada: buňky jurisdikce x rok z let 1990-2014. */
procedura gql conn=icij out=s20_raw;
    query "
        MATCH (e:Entity)
        WHERE e.incorporation_date IS NOT NULL
          AND e.jurisdiction       IS NOT NULL
          AND e.incorporation_date <> '1900-Jan-01'
        WITH split(e.incorporation_date, '-') AS p,
             e.jurisdiction AS jur
        WHERE size(p) = 3
          AND toInteger(p[0]) >= 1990
          AND toInteger(p[0]) <= 2014
        WITH toInteger(p[0]) AS year, jur
        RETURN year, jur AS jurisdiction, count(*) AS n
        ORDER BY year, jurisdiction
    ";
quit;

/* Ponechá jurisdikce s průměrným ročním počtem >= 5. */
procedura sql;
    create table s20_ponechat_jur as
    vybrat jurisdiction, avg(n) as avg_n
    from s20_raw
    group podle jurisdiction
    having avg(n) >= 5;
quit;

procedura sql;
    create table s20 as
    vybrat r.*,
           r.year - 2002 as year_c,
           case když_v r.year >= 2014 pak 1 jinak 0 konec as fatca
    from s20_raw r
    inner join s20_ponechat_jur k
    on r.jurisdiction = k.jurisdiction;
quit;

procedura četnosti data=s20;
    tables jurisdiction;
    název "Section 20 — jurisdictions retained in the panel";
spustit;

/* Poissonův GLMM se smíšenými efekty: pevný trend roku + skok FATCA,
   náhodný intercept podle jurisdikce. */
procedura glimmix data=s20;
    třída jurisdiction;
    model n = year_c fatca / dist=poisson link=log solution;
    random intercept / subject=jurisdiction;
    název "Section 20 — mixed Poisson formation-rate model";
spustit;

/* Seřazené efekty náhodného interceptu by odhalily jurisdikce, které
   systematicky překonávají nebo zaostávají za společným trendem.
   Příkaz SOLUTION v PROC GLIMMIX je vypisuje ve výchozím výstupu výše
   — žebříček ponecháváme na čtenáři. */

In [ ]:
/* Uzavře zprávu PDF a uvolní knihovnu Neo4j. */
ods pdf close;

libname icij clear;

## Reprodukovatelnost a původ dat

- **Zdroj grafových dat:** výzkumná databáze ICIJ *Offshore Leaks*,
  datová sada *Paradise Papers*, poprvé vydaná v listopadu 2017.
  Dostupná na <https://offshoreleaks.icij.org/pages/database>.
  Načtena do sdílené služby `step-neo4j` platformy (Neo4j 5.26
  Community Edition, na úrovni serveru pouze pro čtení) z veřejného
  výpisu na `github.com/neo4j-graph-examples/icij-paradise-papers`.
- **Data OFAC SDN:** veřejný CSV export amerického ministerstva
  financí OFAC Specially Designated Nationals, získaný z veřejného
  preview API ministerstva v dubnu 2026. Soubor `data/ofac_sdn.csv`
  obsahuje 500 kurátorovaných řádků napříč pěti největšími programy
  OFAC podle počtu záznamů. Použit pro dvoustupňové prověření v
  sekci 6.
- **Data OpenSanctions:** snímek *výchozí* konsolidované datové sady
  OpenSanctions z 2026-04-17, stažený z
  `https://data.opensanctions.org/datasets/20260417/default/targets.simple.csv`.
  Uložený soubor `data/opensanctions_default.csv` obsahuje 18,654
  řádků schématu Person a Company, jejichž primární datová sada je
  jedním ze seznamů OFAC SDN, UK HM Treasury, EU financial sanctions,
  Rady bezpečnosti OSN, kanadského, belgického, australského,
  švýcarského nebo jiného národního konsolidovaného sankčního
  seznamu. Aliasy byly vypuštěny, aby soubor zůstal pod 2 MB.
  Licence: ODbL 1.0 (OpenSanctions). Použit pro obohacení v sekci 14.
- **Pořadí daňových rájů:** publikovaná pořadí *Corporate Tax Haven
  Index 2021* organizace Tax Justice Network, sestavená z úvodní
  stránky indexu `https://cthi.taxjustice.net` a tiskové zprávy z
  března 2021 na
  `https://taxjustice.net/press/tax-haven-ranking-shows-countries-setting-global-tax-rules-do-most-to-help-firms-bend-them/`.
  Uložený soubor `data/tax_haven_ranks.csv` uvádí jurisdikce, které se
  objevují v Paradise Papers, s jejich pořadím CTHI a odvozenou váhou
  rizika `[0, 1]`. Licence: CC BY-SA 4.0 (Tax Justice Network). Použit
  pro kompozitní žebříček v sekci 15.
- **Grafové algoritmy:** detekce komunit Louvain a eigenvektorová
  centralita (nejbližší vlastní obdoba PageRanku) počítané pomocí
  `PROC NETWORK` in-process nad seznamy hran staženými přes read-only
  Cypher. Sdílená služba `step-neo4j` platformy je na úrovni serveru
  pouze pro čtení, takže všechny grafové algoritmy běží ve workspace
  podu, nikoli přes zápisové operace Neo4j Graph Data Science.
- **Statistické metody:** `PROC LIFETEST` používá Kaplan-Meierův
  odhad se stratifikovanými testy log-rank, Wilcoxon a Tarone-Ware.
  `PROC PHREG` přizpůsobuje Coxův model proporcionálních rizik přes
  Breslowovo ošetření shod pomocí wrapperu Python/statsmodels.
  `PROC LOGISTIC` přizpůsobuje binární logistickou regresi metodou
  maximální věrohodnosti.
- **Metody skládání rizika:** kompozitní skóre vlivu ze sekce 11
  normalizuje stupeň, log-PageRank, jurisdikční šíři a roky působení
  na `[0, 1]` a sčítá je s pevnými váhami (30/30/20/20). Kompozitní
  skóre rizika subjektu ze sekce 15 normalizuje omezený počet
  funkcionářů, log-PageRank, váhu rizika CTHI a binární příznak
  sankcionovaného funkcionáře a sčítá je se stejnými váhami 0.25 každá.

Kompletní analýza je reprodukovatelná ze smoke-test skriptu v této
složce: `./smoke.jenner`. Jeho spuštění od začátku do konce vůči
sdílené službě `step-neo4j` (s nastavenými `JENNER_NEO4J_HOST` a
`JENNER_NEO4J_PASS`, což platforma za vás v každém workspace podu
udělá) trvá zhruba dvě minuty a ověřuje, že každý dotaz a každá
procedura — včetně pěti nových sekcí přidaných vedle stávající
pipeline — vrací očekávaný výstup nad reálným grafem se 163,435 uzly.